In [9]:
import os
from dotenv import load_dotenv
from langfuse import get_client
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from smolagents import CodeAgent, LiteLLMModel, GoogleSearchTool, VisitWebpageTool

load_dotenv(override=True)

# Tracing
langfuse = get_client()
print("Langfuse ready:", langfuse.auth_check())
SmolagentsInstrumentor().instrument()

# Shared model for both agents
model = LiteLLMModel(
    model_id="gemini/gemini-3.5-flash",
    api_key=os.environ["GEMINI_API_KEY"],
    timeout=60,
    num_retries=0,
)

Attempting to instrument while already instrumented


Langfuse ready: True


Initialize the sub agent who browse and retrieve information from internet websites

In [ ]:
# Sub-agent: only browses the web, no extra imports
web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(max_output_length=5000),
        ],
    name="web_agent",
    description=(
        "Searches the web and reads webpages."
        "Use it for any information that must be found online."
        "Give it a precise, self-contained question: it has no access to the manager's context."
    ),
    max_steps=6,
)

Initialize the manager of the sub-agent

In [ ]:
# Manager: delegates web lookups, never browses itself
manager_agent = CodeAgent(
    model=model,
    tools=[],
    managed_agents=[web_agent],
    max_steps=6,
)

# Check the multi-agent setup
manager_agent.visualize()

CodeAgent | gemini/gemini-3.5-flash
├── ✅ Authorized imports: []
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name         ┃ Description                                   ┃ Arguments                                    ┃
│   ┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ final_answer │ Provides a final answer to the given problem. │ answer (`any`): The final answer to the      │
│   │              │                                               │ problem                                      │
│   └──────────────┴───────────────────────────────────────────────┴──────────────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | gemini/gemini-3.5-flash
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Searches the web and reads webpages. Use it for any information that must be found 
        │   online. Give it a precise, self-contained question: it has no access to the manager's context.
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name          ┃ Description                              ┃ Arguments                                ┃
            ┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search    │ Performs a google web search for your    │ query (`string`): The search query to    │
            │               │ query then returns a string of the top   │ perform.                                 │
            │               │ search results.                          │ filter_year (`integer`): Optionally      │
            │               │                                          │ restrict results to a certain year       │
            │ visit_webpage │ Visits a webpage at the given url and    │ url (`string`): The url of the webpage   │
            │               │ reads its content as a markdown string.  │ to visit.                                │
            │               │ Use this to browse webpages.             │                                          │
            │ final_answer  │ Provides a final answer to the given     │ answer (`any`): The final answer to the  │
            │               │ problem.                                 │ problem                                  │
            └───────────────┴──────────────────────────────────────────┴──────────────────────────────────────────┘